In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import Dataset
from transformers import (
    AutoImageProcessor,
    ResNetForImageClassification,
    ResNetConfig,
    TrainingArguments,
    Trainer
)

# 隱藏討厭的版本更新 Warning 提示
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================
# 1. M3 Mac GPU / CUDA 加速設定
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"🚀 Hugging Face 管線目前使用的運作裝置: {device}")

# 載入標準 ResNet18 的影像前處理器 (負責自動將圖片處理成模型要的形狀)
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")

# ==========================================
# 2. 🌟 建立 Hugging Face 專用 Dataset (模擬真實 4 類別二維陣列)
# ==========================================
def generate_dummy_data(num_samples=20):
    data_list = []
    for _ in range(num_samples):
        # 1. 模擬 Ben 之後給你的真實彩色照片 (512x512x3)
        fake_image = np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)
        pil_img = Image.fromarray(fake_image)

        # 2. 🌟 完美模擬你之後會得到的 [[1,2,3,4...]*16] 二維陣列標籤
        fake_grid_labels = np.random.randint(1, 5, (16, 16))

        data_list.append({
            "image": pil_img,
            "grid_label": fake_grid_labels.tolist() # 轉成 list 格式以利 HF Dataset 封裝
        })
    return Dataset.from_list(data_list)

raw_dataset = generate_dummy_data(20)
print("✅ 成功建立 Hugging Face 格式的虛擬資料集！")

# ==========================================
# 3. 🌟 關鍵：將二維陣列轉換成 Hugging Face 模型認得的 One-Hot 標籤
# ==========================================
def transform_function(examples):
    # 使用 HF 前處理器處理圖片，欄位名稱會自動變成 'pixel_values'
    inputs = processor([img.convert("RGB") for img in examples["image"]], return_tensors="pt")

    batch_labels = []
    for grid in examples["grid_label"]:
        grid_array = np.array(grid) # 讀入 (16, 16) 矩陣，數值為 1~4

        # 建立一個 One-hot 矩陣，形狀為 (16, 16, 4)
        # 類別 1 對應索引 0，類別 2 對應索引 1... 依此類推
        one_hot = np.zeros((16, 16, 4), dtype=np.float32)
        for c in range(1, 5):
            one_hot[:, :, c-1] = (grid_array == c).astype(np.float32)

        # 💡 關鍵：壓扁成符合一維 1024 (16x16x4) 個模型輸出的多標籤規格
        batch_labels.append(one_hot.flatten())

    inputs["labels"] = torch.tensor(batch_labels)
    return inputs

# 使用 Dataset.map 進行批次預處理
processed_dataset = raw_dataset.map(transform_function, batched=True, remove_columns=raw_dataset.column_names)

# ==========================================
# 4. 定義模型 (總輸出數量 = 16 x 16 x 4 = 1024)
# ==========================================
config = ResNetConfig.from_pretrained("microsoft/resnet-18", num_labels=1024)
model = ResNetForImageClassification.from_pretrained(
    "microsoft/resnet-18",
    config=config,
    ignore_mismatched_sizes=True
).to(device)

# 💡 告訴模型這是一個多標籤分類任務，內部會自動套用 BCEWithLogitsLoss
model.config.problem_type = "multi_label_classification"

# ==========================================
# 5. 使用 Hugging Face Trainer 進行訓練
# ==========================================
training_args = TrainingArguments(
    output_dir="../hf-grid-model",
    num_train_epochs=3,                # 測試跑 3 個 Epoch
    per_device_train_batch_size=4,
    logging_steps=1,
    remove_unused_columns=False,       # 💡 關鍵：避免 HF 自動刪除我們自訂的 pixel_values 和 labels
    use_cpu=False if torch.backends.mps.is_available() or torch.cuda.is_available() else True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
)

print("\n🚀 啟動 Hugging Face Trainer 進行空間網格模型訓練...")
trainer.train()

# ==========================================
# 6. 🌟 推理測試與還原 [[1,2,3,4...]*16] 二維陣列
# ==========================================
print("\n🔮 訓練完成！模擬拿一張新圖進行預測並還原成 1~4 的二維陣列...")
model.eval()

# 模擬測試：拿原始資料集的第一張圖 (Row 0)
test_raw_img = raw_dataset[0]["image"]
inputs = processor(test_raw_img, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# 拿取模型輸出的 1024 個分數並還原形狀為 (16, 16, 4)
logits = outputs.logits.squeeze(0).cpu().numpy()
spatial_logits = logits.reshape(16, 16, 4)

# 💡 後處理核心：在最後一個維度（4個類別通道）中，找出得分最高的那一個索引 (0~3)，然後 +1 變回 1~4
grid_predictions = np.argmax(spatial_logits, axis=-1) + 1

# ==========================================
# 7. 使用 Matplotlib 畫出對照圖
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左邊：輸入的圖片
axes[0].imshow(test_raw_img)
axes[0].set_title("1. Input Image (512x512)", fontsize=12)
for x in np.linspace(0, 512, 17):
    axes[0].axvline(x, color='white', linestyle='--', alpha=0.5, linewidth=1)
    axes[0].axhline(x, color='white', linestyle='--', alpha=0.5, linewidth=1)

# 右邊：還原後的 1~4 類別彩色網格
cmap = plt.cm.get_cmap('Set1', 4)
im = axes[1].imshow(grid_predictions, cmap=cmap, vmin=0.5, vmax=4.5)
axes[1].set_title("2. Hugging Face Model Output (16x16 Matrix)", fontsize=12)

# 在格子中央打上 1, 2, 3, 4 的數字
for i in range(16):
    for j in range(16):
        axes[1].text(j, i, str(grid_predictions[i, j]), ha="center", va="center", color="white", fontsize=9)

axes[1].set_xticks(np.arange(16))
axes[1].set_yticks(np.arange(16))
cbar = fig.colorbar(im, ax=axes[1], ticks=[1, 2, 3, 4], fraction=0.046, pad=0.04)
cbar.ax.set_yticklabels(['Category 1', 'Category 2', 'Category 3', 'Category 4'])

plt.tight_layout()
plt.show()

# 顯示最終產出的二維陣列結果
print("\n--- 📝 最終輸出給你的 16x16 4類別整數二維陣列 ---")
print(grid_predictions)

🚀 Hugging Face 管線目前使用的運作裝置: mps


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ 成功建立 Hugging Face 格式的虛擬資料集！


Map: 100%|██████████| 20/20 [00:00<00:00, 143.15 examples/s]
Some weights of ResNetForImageClassification were not initialized from the model checkpoint at microsoft/resnet-18 and are newly initialized because the shapes did not match:
- classifier.1.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([1024]) in the model instantiated
- classifier.1.weight: found shape torch.Size([1000, 512]) in the checkpoint and torch.Size([1024, 512]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 啟動 Hugging Face Trainer 進行空間網格模型訓練...
